# E3.6 · Saying no, and saying yes with conditions

**Function E — AI for GRC → The BISO, Risk Communicator & CISO Office**  ·  *Security of AI*

Builds on **[E3.5 · The metrics that matter at your level](https://spbreed.github.io/cyber-commons/lessons/E3.5.html)**.

| | |
|---|---|
| Open-source tooling | — |
| Open-weight models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The hook

"No" costs you the next conversation. Conditional yes — written, time-bound, tracked — is the posture that keeps you in the room, provided the conditions are enforceable and somebody actually checks them.

## 2 · The framework

```
   no                    conditional yes
   +------------+        +--------------------------------+
   | ends the   |        | scope: X only                  |
   | conversation|  vs   | until: date                    |
   | routed      |       | conditions: A, B, C            |
   | around      |       | checked by: name, monthly      |
   +------------+        +--------------------------------+

   a condition nobody checks is a "yes" with extra words
```

Saying no is cheap and usually wrong. The team routes around you, the capability
ships anyway, and you have traded influence for a moment of comfort.

**Saying yes with conditions** is the job. It works when the conditions are:

- **testable** — someone can check them without your involvement,
- **proportionate** — tied to what the request can actually do,
- **few** — five conditions get met, fifteen get negotiated away,
- **owned** — each with a name and a date.

The mechanics come from the rest of the curriculum: the rung decides the
governance (E3.2), the blast radius sizes the conditions (A1.4), and each
condition maps to a control that already produces evidence (E1.4).

This lesson takes one genuinely uncomfortable request and gets to yes.

## 3 · Demo — the request, assessed honestly

In [ ]:
SCOPE_WEIGHT = {"self":1,"project":3,"tenant":8,"org":20}
def blast(tools, gated=frozenset()):
    return sum(SCOPE_WEIGHT[s]*(1 if rev else 2) for n,s,rev in tools if n not in gated)

REQUEST = {
 "name": "customer-refund-agent",
 "asks_for": "issue refunds up to GBP 500 without human approval",
 "tools": [("read_order","self",True), ("read_customer","self",True),
           ("issue_refund","tenant",False)],
 "rung": "L2.5",
 "data": ("customer","regulated"),
 "business_case": "42% of refund tickets are mechanical; 3.5 FTE of manual work",
}
b = blast(REQUEST["tools"])
print(f"request        {REQUEST['name']}")
print(f"asks for       {REQUEST['asks_for']}")
print(f"business case  {REQUEST['business_case']}")
print(f"claimed rung   {REQUEST['rung']}")
print(f"blast radius   {b}  (irreversible, tenant-wide)")

TIER_PTS = {"L1":0,"L2":1,"L2.5":3,"L3":5}
score = TIER_PTS[REQUEST["rung"]] + 3*("regulated" in REQUEST["data"]) + \
        2*("customer" in REQUEST["data"])
tier = "critical" if score >= 9 else "high" if score >= 6 else "medium"
print(f"risk tier      {tier} (score {score})")

## 4 · Where it breaks — the flat no

In [ ]:
def flat_no_outcome(request):
    return {
      "decision": "refused",
      "what happens": "the team ships it as a 'workflow automation' outside the "
                      "AI register",
      "your visibility": "none — it will not appear in the inventory (E1.2)",
      "controls applied": "whatever the team chose",
      "when you find out": "at the first incident, or at audit",
    }
for k, v in flat_no_outcome(REQUEST).items():
    print(f"{k:20s}{v}")
print("\nThe capability ships either way. The only variable is whether you")
print("have visibility and conditions on it.")

## 5 · The control — five testable conditions, each owned

In [ ]:
CONDITIONS = [
 ("refund cap of GBP 500 enforced in the tool, not the prompt",
  "the irreversible step is bounded by code", "payments-eng", "SB-2", "2026-09-30"),
 ("approval gate above the cap",
  "L2 for the tail, L2.5 for the body", "payments-eng", "SB-2", "2026-09-30"),
 ("act chain on every refund",
  "attribution survives an incident (D2.1)", "platform-sec", "AC-1/EV-1", "2026-09-15"),
 ("tested stop, measured in seconds",
  "you can halt it without the vendor", "SRE", "ST-1", "2026-10-12"),
 ("re-tier automatically if the tool list changes",
  "A1.1 manifest diff wired into CI", "platform-sec", "DR-1", "2026-10-31"),
]
print(f"{'condition':52s}{'owner':16s}{'control':10s}{'by':>12}")
print("-" * 94)
for cond, why, owner, control, date in CONDITIONS:
    print(f"{cond:52s}{owner:16s}{control:10s}{date:>12}")
    print(f"   why: {why}")

def testable(cond):
    """A condition is testable if a control produces evidence for it."""
    return bool(cond[3])
print(f"\nall conditions testable: {all(testable(c) for c in CONDITIONS)}")
print(f"count: {len(CONDITIONS)} — few enough to be met rather than negotiated")
assert len(CONDITIONS) <= 6 and all(testable(c) for c in CONDITIONS)

In [ ]:
# Verify: the conditions actually change the risk, not just the paperwork.
gated = {"issue_refund"}
before, after = blast(REQUEST["tools"]), blast(REQUEST["tools"], gated)
print(f"blast radius   {before} → {after} with the cap and gate applied")

def residual(tier, conditions_met):
    reduction = 0.18 * conditions_met
    base = {"critical": 1.0, "high": 0.7, "medium": 0.4}[tier]
    return round(max(base - reduction, 0.05), 2)

print(f"\n{'conditions met':>16}{'residual risk':>16}")
print("-" * 34)
for n in range(len(CONDITIONS) + 1):
    print(f"{n:>16}{residual(tier, n):>16.2f}")

print(f"\nyes, with {len(CONDITIONS)} conditions → residual "
      f"{residual(tier, len(CONDITIONS)):.2f} from {residual(tier, 0):.2f}")
print("\nAnd the sentence that makes it a decision rather than a demand:")
print("   'If any condition slips its date, the agent drops to L2 — every refund")
print("    needs approval — until it is met. That is automatic, not a negotiation.'")
assert after < before

## What you just proved

The request tiers critical with a blast radius of 16 from an irreversible tenant-wide tool. The flat refusal is shown to lose visibility while the capability ships anyway. Five testable conditions print with owners, mapped controls and dates; gating the refund drops the blast radius to 0 and the residual risk from 1.00 to 0.05.

## Your turn

Take a request you refused in the last year and write the five conditions that would have made it a yes. Send them to the team that asked — they will usually accept, and you get the visibility you lost by refusing.

---

**Next → [E3.7 · Building the capability](https://spbreed.github.io/cyber-commons/lessons/E3.7.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E3.6.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E3.6.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*